# 🐼 Pandas Data Cleaning & Feature Wrangling Lab
**Interactive Machine Learning & Data Science Notebook**  
*Author: Muhammad Zabiullah ([@mzb000](https://github.com/mzb000))*  

---

## 📌 Objectives
1. Master missing data detection and modern imputation strategies.
2. Detect and remediate outliers using **IQR** (Interquartile Range) and **Z-Score**.
3. Perform multi-level grouped aggregations with `.agg()` and pivot tables.
4. Optimize DataFrame memory footprint for large datasets.

### Step 1: Missing Data Remediation
Real-world data is noisy and full of nulls (`NaN`). Dropping rows causes loss of critical training signal. Modern machine learning relies on principled imputation.

In [ ]:
import numpy as np
import pandas as pd

raw_data = {
    "customer_id": [101, 102, 103, 104, 105, 106],
    "age": [25, np.nan, 32, 45, np.nan, 29],
    "income": [50000, 62000, np.nan, 85000, 92000, np.nan],
    "city": ["Lahore", "Karachi", "Islamabad", np.nan, "Karachi", "Lahore"]
}

df = pd.DataFrame(raw_data)
print("Missing Values Summary:")
print(df.isnull().sum())

# Imputation:
# 1. Median for skewed continuous data (Age & Income)
df["age_imputed"] = df["age"].fillna(df["age"].median())
df["income_imputed"] = df["income"].fillna(df["income"].median())

# 2. Mode for categorical features (City)
df["city_imputed"] = df["city"].fillna(df["city"].mode()[0])

print("\nCleaned DataFrame:")
print(df[["customer_id", "age_imputed", "income_imputed", "city_imputed"]])

### Step 2: Outlier Detection using the IQR Method
Values that fall outside $[Q_1 - 1.5 \times IQR, Q_3 + 1.5 \times IQR]$ are flagged as outliers.
Rather than discarding observations, **Winsorization** (clipping) preserves dataset size while protecting gradient descent from exploding gradients.

In [ ]:
np.random.seed(42)
metric_data = np.concatenate([
    np.random.normal(loc=100, scale=10, size=200),
    [250, 275, -20]  # Extreme anomalous outliers
])

metric_df = pd.DataFrame({"metric": metric_data})

q1 = metric_df["metric"].quantile(0.25)
q3 = metric_df["metric"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)

outliers = metric_df[(metric_df["metric"] < lower_bound) | (metric_df["metric"] > upper_bound)]
print(f"IQR Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Total Outliers Detected: {len(outliers)}")

# Robust Capping (Winsorization)
metric_df["metric_clean"] = metric_df["metric"].clip(lower=lower_bound, upper=upper_bound)
print("Original Range:", metric_df["metric"].min(), "to", metric_df["metric"].max())
print("Capped Range  :", metric_df["metric_clean"].min(), "to", metric_df["metric_clean"].max())

### Step 3: Multi-Level GroupBy & Feature Aggregation
Aggregating business metrics across segments.

In [ ]:
transactions = pd.DataFrame({
    "store": ["North", "North", "South", "South", "East", "East"],
    "category": ["Electronics", "Groceries", "Electronics", "Groceries", "Electronics", "Groceries"],
    "revenue": [4500, 1200, 5200, 950, 3100, 1800],
    "transactions": [45, 120, 50, 95, 30, 140]
})

# Multi-metric aggregation
summary = transactions.groupby("store").agg({
    "revenue": ["sum", "mean"],
    "transactions": ["sum"]
})
print("Store Performance Summary:")
print(summary)

### Step 4: Memory Optimization Techniques
Reducing memory consumption by up to 80% on tabular data.

In [ ]:
# Demonstrating Category Datatype
status_series = pd.Series(["Completed", "Pending", "Failed"] * 100000)

initial_mem = status_series.memory_usage(deep=True) / 1024**2
category_series = status_series.astype("category")
optimized_mem = category_series.memory_usage(deep=True) / 1024**2

print(f"Initial Object Memory  : {initial_mem:.2f} MB")
print(f"Optimized Category RAM : {optimized_mem:.2f} MB")
print(f"Memory Reduction       : {((initial_mem - optimized_mem) / initial_mem) * 100:.1f}%")